# Assistente per il dizionario codici

Notebook **autonomo**: legge lo stesso `xml.zip` di `Quadratura_Fatture.ipynb`, ma non fa quadratura.
Serve a rispondere a una domanda sola: **quali `CodiceTipoXML` arrivano davvero dagli XML, e a quale
campo canonico vanno fatti corrispondere?**

Nella quadratura un tipo non presente in `RegoleCodiciXML` per quel `CFOR` viene scartato in silenzio
(sezione `[12]`). Compilare il foglio a mano significa aprire migliaia di XML e indovinare se `FORN`,
`COD_F`, `ARTICOLO FORNITORE` o `MINSAN` sono EAN, codice cliente o codice fornitore.

## Come decide

Per ogni coppia `CFOR + CodiceTipoXML` incrocia **tre segnali indipendenti**:

| # | Segnale | Esempio |
|---|---|---|
| 1 | **nome del tipo** | `EAN`/`GTIN`/`BARCODE` → `EAN`; `FORN`/`SUPPLIER` → `CODICE_FORNITORE`; `CLI`/`CUSTOMER` → `CODICE_CLIENTE` |
| 2 | **forma dei valori** | GTIN 8/12/13/14 con checksum valido → `EAN`; codici alfanumerici con trattini → probabile codice articolo (ma non dice da che parte) |
| 3 | **riscontro sui dati aziendali** | il valore normalizzato esiste in `CEAN`? in `CART`? in `CARF`/`CALA`? |

I segnali si sommano con pesi diversi e producono un `CampoCanonico`, una
`RegolaNormalizzazione` e una `Confidenza` (`ALTA` / `MEDIA` / `BASSA` / `NESSUNA`). Quando i segnali
sono assenti o in conflitto, il tipo **non viene proposto**: finisce nel foglio `DaVerificare`, con
tutte le evidenze raccolte, per la decisione umana.

## Cosa produce

`export/proposte_dizionario_codici.xlsx` con i fogli:

| Foglio | Contenuto |
|---|---|
| `ProposteXML` | righe nello schema di `RegoleCodiciXML`, pronte da copiare dopo verifica |
| `ProposteCarico` | righe simmetriche per `RegoleCodiciCarico` (evita le «regole diverse fra i due lati») |
| `DaVerificare` | tipi non classificati, con segnali in conflitto o senza `CFOR` |
| `DettaglioTipi` | tutte le evidenze e i punteggi, per audit |
| `FornitoriDaMappare` | fatture di cui non si è trovato il `CFOR` |
| `RegoleInutilizzate` | regole presenti nel dizionario ma mai osservate negli XML |
| `AnomalieDizionario` | `CampoCanonico`/regole non riconosciute e chiavi duplicate |
| `Riepilogo` | conteggi dell'esecuzione |

> **L'assistente non scrive mai `Dizionario codici.xlsx`.** Propone soltanto: la scelta finale resta a
> chi conosce gli articoli del fornitore. Il limite strutturale è che l'XML **non contiene il CFOR**:
> viene ricavato dai carichi (`CEAS + NDOGIAA`, come nella sezione `[11]` della quadratura), con
> ripiego sulla partita IVA e sulla mappa manuale `MAPPA_PIVA_CFOR` della cella `[1]`.

Eseguire le celle in ordine. Logica, pesi e limiti sono descritti in
[`docs/ASSISTENTE_DIZIONARIO.md`](docs/ASSISTENTE_DIZIONARIO.md).

## [1] Configurazione e sorgenti

Stessi percorsi OneLake della quadratura, più tre parametri propri: `MAPPA_PIVA_CFOR` (per i
fornitori assenti dai carichi), le soglie di riscontro/confidenza e `ESPORTA_PROPOSTE`. Con
`MODALITA_LOCALE = True` si leggono i file da una cartella del disco, senza `notebookutils`.

In [ ]:
# ==============================================================
# [1] CONFIGURAZIONE E SORGENTI
# ==============================================================

import os
import re
import unicodedata
import zipfile
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import PurePosixPath

import pandas as pd


# In Fabric display() e' disponibile; in esecuzioni locali no.
try:
    display
except NameError:
    def display(*oggetti, **kwargs):
        for oggetto in oggetti:
            print(oggetto)


# Stessa cartella OneLake del notebook di quadratura.
cartella = (
    "abfss://d072e408-7302-41de-a976-f6b6b626177e"
    "@onelake.dfs.fabric.microsoft.com/"
    "401c1a6a-fa86-43ab-a153-01e4fa9de79f/Files"
)

# True -> i file si leggono da CARTELLA_LOCALE, senza OneLake.
MODALITA_LOCALE = False
CARTELLA_LOCALE = "./dati"

ZIP_FILE = "xml.zip"
zip_locale_python = "/tmp/assistente_dizionari_xml.zip"
zip_locale_fs = "file:" + zip_locale_python

CARICHI_FILE = "TAB_CM_Fatture_BI.xlsx"
CARICHI_SHEET = "TAB_CM_Fatture_BI"
EAN_FILE = "TAB_ANAGRAFICA_EAN.xlsx"
EAN_SHEET = "TAB_CM_ANAGRAFICA_EAN_BI"
CARF_FILE = "TAB_ANAGRAFICA_CARF.xlsx"
CARF_SHEET = "TAB_CM_ANAGRAFICA_CARF_BI"

DIZIONARIO_FILE = "Dizionario codici.xlsx"
dizionario_locale_python = "/tmp/assistente_Dizionario_codici.xlsx"
dizionario_locale_fs = "file:" + dizionario_locale_python

NOME_FOGLIO_CODICI_XML = "RegoleCodiciXML"
NOME_FOGLIO_CODICI_CARICO = "RegoleCodiciCarico"

# L'assistente NON modifica il dizionario: scrive solo un file di proposte.
ESPORTA_PROPOSTE = True
OUTPUT_FILE = "proposte_dizionario_codici.xlsx"

# Se un fornitore non e' nei carichi, qui si puo' associare la partita IVA
# (o la denominazione) al CFOR usato nel gestionale:
#     MAPPA_PIVA_CFOR = {"IT01234567890": "F001"}
MAPPA_PIVA_CFOR = {}

# Soglie e pesi: dettagli in docs/ASSISTENTE_DIZIONARIO.md
SOGLIA_RISCONTRO = 0.40
MIN_RISCONTRI = 2
MAX_ESEMPI = 4
SOGLIA_ALTA = 7.0
SOGLIA_MEDIA = 4.0
MARGINE_MINIMO = 1.5

CAMPI_CANONICI = ("EAN", "CODICE_CLIENTE", "CODICE_FORNITORE")
REGOLE_NOTE = {
    "RIMUOVI_SEPARATORI", "RIMUOVI_ZERI", "NESSUNA_REGOLA",
    "TESTO_MAIUSCOLO", "SOLO_NUMERI",
    "ESTRAI_PRIMA_STRINGA", "ESTRAI_ULTIMA_STRINGA",
}
COLONNE_PROPOSTA_XML = [
    "CFOR", "Denominazione", "CodiceTipoXML",
    "CampoCanonico", "RegolaNormalizzazione",
]
COLONNE_PROPOSTA_CARICO = [
    "CFOR", "Denominazione", "CampoCanonico", "RegolaNormalizzazione",
]


def percorso(nome_file):
    base = CARTELLA_LOCALE if MODALITA_LOCALE else cartella
    return f"{base}/{nome_file}"


def copia_in_locale(sorgente, destinazione_fs, destinazione_python):
    """
    Copia da OneLake in locale (come la quadratura), oppure usa il
    file gia' presente sul disco in modalita' locale.
    """
    if MODALITA_LOCALE:
        if not os.path.exists(sorgente):
            raise FileNotFoundError(f"File non trovato: {sorgente}")
        return sorgente

    if notebookutils.fs.exists(destinazione_fs):
        notebookutils.fs.rm(destinazione_fs)

    if not notebookutils.fs.cp(sorgente, destinazione_fs):
        raise RuntimeError(f"Impossibile copiare in locale: {sorgente}")

    return destinazione_python


def leggi_excel(nome_file, nome_foglio):
    """
    Legge un Excel di ingresso. Con nome_foglio=None si leggono tutti
    i fogli del dizionario (sheet_name=None), come nella quadratura.
    """
    if nome_foglio is None:
        locale_fs = dizionario_locale_fs
        locale_py = dizionario_locale_python
    else:
        locale_fs = "file:/tmp/assistente_" + nome_file.replace(" ", "_")
        locale_py = locale_fs[len("file:"):]

    locale = copia_in_locale(percorso(nome_file), locale_fs, locale_py)
    return pd.read_excel(locale, sheet_name=nome_foglio, dtype=str)


print("Configurazione caricata. Modalità locale:", MODALITA_LOCALE)

## [2] Lettura degli XML: una riga per codice articolo

Come nella sezione `[2]` della quadratura: si copia lo ZIP in locale, si tengono solo i documenti
`FPR12` e si tolgono gli `<Allegati>`. Per ogni `CodiceArticolo` si conserva il tipo **così come
scritto dal fornitore** (`CodiceTipoXML`) e la sua forma normalizzata per i confronti
(`TipoChiave`). Viene ricreato anche il tipo virtuale `RiferimentoAmministrazione`, che la
quadratura genera dalle linee: chi compila il dizionario deve vederlo, altrimenti resta invisibile.

`PartitaIVA` si legge dall'intestazione; in mancanza, si ricava dal nome del file SDI.

In [ ]:
# ==============================================================
# [2] LETTURA DEGLI XML: UNA RIGA PER OGNI CODICE ARTICOLO
# ==============================================================

SEPARATORI = " -/._:"
CIFRE = "0123456789"


def testo(elemento):
    """Testo pulito di un nodo XML; None se assente o vuoto."""
    if elemento is not None and elemento.text:
        return elemento.text.strip() or None
    return None


def normalizza_testo(valore):
    """
    Pulizia di base: caratteri non stampabili, trim, maiuscolo.
    Equivalente a in_testo/valore_pulito della quadratura.
    """
    if valore is None:
        return None
    try:
        if pd.isna(valore):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(valore, float) and valore.is_integer():
        valore = int(valore)
    base = str(valore).replace("\u00a0", " ")
    base = "".join(c for c in base if c.isprintable())
    base = base.strip().upper()
    return base or None


def ceas_da_nome_file(nome_file):
    """Nome del file senza percorso e senza estensione."""
    return PurePosixPath(nome_file).stem


def piva_da_ceas(ceas):
    """Ripiego: il nome del file SDI contiene la partita IVA del cedente."""
    if not ceas:
        return None
    prima = ceas.split("_")[0].upper()
    return prima if re.fullmatch(r"[A-Z]{2}[A-Z0-9]{6,}", prima) else None


righe_codici = []
righe_documenti = []
errori = []

zip_python = copia_in_locale(
    percorso(ZIP_FILE), zip_locale_fs, zip_locale_python
)

with zipfile.ZipFile(zip_python, mode="r") as archivio_zip:

    danneggiato = archivio_zip.testzip()
    if danneggiato is not None:
        raise zipfile.BadZipFile(
            f"File danneggiato nello ZIP: {danneggiato}"
        )

    for info_xml in archivio_zip.infolist():

        if info_xml.is_dir() or not info_xml.filename.lower().endswith(".xml"):
            continue

        nome_file = info_xml.filename

        try:
            root = ET.fromstring(archivio_zip.read(info_xml))

            if root.attrib.get("versione", "").upper() != "FPR12":
                continue

            # Gli allegati (PDF) non servono: si eliminano subito.
            for body in root.findall(".//{*}FatturaElettronicaBody"):
                for allegati in body.findall("{*}Allegati"):
                    body.remove(allegati)

            cedente = root.find(
                ".//{*}CedentePrestatore/{*}DatiAnagrafici"
            )
            denominazione = None
            partita_iva = None

            if cedente is not None:
                anagrafica = cedente.find("{*}Anagrafica")
                if anagrafica is not None:
                    denominazione = testo(
                        anagrafica.find("{*}Denominazione")
                    )
                    if not denominazione:
                        denominazione = " ".join(
                            v for v in (
                                testo(anagrafica.find("{*}Nome")),
                                testo(anagrafica.find("{*}Cognome")),
                            ) if v
                        ) or None

                partita_iva = testo(
                    cedente.find("{*}IdFiscaleIVA/{*}IdCodice")
                )

            ceas = ceas_da_nome_file(nome_file)
            if not partita_iva:
                partita_iva = piva_da_ceas(ceas)

            bodies = root.findall(".//{*}FatturaElettronicaBody")

            for indice_body, body in enumerate(bodies, start=1):

                numero_fattura = testo(
                    body.find(
                        "./{*}DatiGenerali/"
                        "{*}DatiGeneraliDocumento/{*}Numero"
                    )
                )
                chiave_documento = f"{nome_file}#{indice_body}"

                righe_documenti.append({
                    "ChiaveDocumento": chiave_documento,
                    "FileXML": nome_file,
                    "CEAS": ceas,
                    "IndiceBodyXML": indice_body,
                    "NumeroFattura": numero_fattura,
                    "Denominazione": denominazione,
                    "PartitaIVA": partita_iva,
                })

                for dettaglio in body.findall(
                    "./{*}DatiBeniServizi/{*}DettaglioLinee"
                ):
                    numero_linea = testo(
                        dettaglio.find("./{*}NumeroLinea")
                    )
                    blocchi = dettaglio.findall("./{*}CodiceArticolo")

                    for progressivo, blocco in enumerate(
                        blocchi, start=1
                    ):
                        tipo_grezzo = testo(
                            blocco.find("./{*}CodiceTipo")
                        )
                        righe_codici.append({
                            "ChiaveDocumento": chiave_documento,
                            "FileXML": nome_file,
                            "NumeroLinea": numero_linea,
                            "Denominazione": denominazione,
                            "ProgressivoCodiceXML": progressivo,
                            "Origine": "CodiceArticolo",
                            "CodiceTipoXML": (
                                tipo_grezzo or "(senza tipo)"
                            ),
                            "TipoChiave": (
                                normalizza_testo(tipo_grezzo) or ""
                            ),
                            "CodiceValoreXML": testo(
                                blocco.find("./{*}CodiceValore")
                            ),
                        })

                    # Tipo virtuale creato anche dalla quadratura [3].
                    riferimento = testo(
                        dettaglio.find(
                            "./{*}RiferimentoAmministrazione"
                        )
                    )
                    if riferimento:
                        righe_codici.append({
                            "ChiaveDocumento": chiave_documento,
                            "FileXML": nome_file,
                            "NumeroLinea": numero_linea,
                            "Denominazione": denominazione,
                            "ProgressivoCodiceXML": len(blocchi) + 1,
                            "Origine": "VIRTUALE",
                            "CodiceTipoXML": "RiferimentoAmministrazione",
                            "TipoChiave": normalizza_testo(
                                "RiferimentoAmministrazione"
                            ),
                            "CodiceValoreXML": riferimento,
                        })

        except Exception as errore:
            errori.append({"File": nome_file, "Errore": str(errore)})


COLONNE_CODICI = [
    "ChiaveDocumento", "FileXML", "NumeroLinea", "Denominazione",
    "ProgressivoCodiceXML", "Origine", "CodiceTipoXML", "TipoChiave",
    "CodiceValoreXML",
]
COLONNE_DOCUMENTI = [
    "ChiaveDocumento", "FileXML", "CEAS", "IndiceBodyXML",
    "NumeroFattura", "Denominazione", "PartitaIVA",
]

df_codici = pd.DataFrame(righe_codici, columns=COLONNE_CODICI)
df_documenti = pd.DataFrame(
    righe_documenti, columns=COLONNE_DOCUMENTI
)
df_errori = pd.DataFrame(errori, columns=["File", "Errore"])

print(
    f"Documenti FPR12: {len(df_documenti):,} - "
    f"codici articolo: {len(df_codici):,} - "
    f"tipi distinti: {df_codici['TipoChiave'].nunique():,}"
)

if not df_errori.empty:
    print(f"XML non elaborati: {len(df_errori)}")
    display(df_errori)

## [3] Attribuzione del `CFOR`

È il punto delicato: **`RegoleCodiciXML` è chiavato per `CFOR + CodiceTipoXML`, ma l'XML non porta
il `CFOR`**. Si usa quindi la stessa regola della sezione `[11]` della quadratura: coppia
`CEAS + NDOGIAA` dai carichi. Ripieghi, in ordine: mappa manuale `MAPPA_PIVA_CFOR` (prioritaria),
`CEAS` con un solo fornitore, partita IVA/denominazione per il foglio `FornitoriDaMappare`.

Senza `CFOR` le righe non possono essere proposte: vengono comunque analizzate e segnalate, perché
un fornitore nuovo è esattamente il caso in cui serve una riga nuova nel dizionario.

In [ ]:
# ==============================================================
# [3] ATTRIBUZIONE DEL CFOR (IL DIZIONARIO E' PER FORNITORE)
# ==============================================================

df_carichi = leggi_excel(CARICHI_FILE, CARICHI_SHEET)

for colonna in ("CEAS", "NDOGIAA", "CFOR"):
    if colonna not in df_carichi.columns:
        raise ValueError(
            f"Colonna {colonna} assente nei carichi: "
            "serve per attribuire il CFOR"
        )

mappa_fattura = defaultdict(set)
mappa_ceas = defaultdict(set)

for ceas, numero, cfor in zip(
    df_carichi["CEAS"],
    df_carichi["NDOGIAA"],
    df_carichi["CFOR"],
):
    k_ceas = normalizza_testo(ceas)
    k_numero = normalizza_testo(numero)
    k_cfor = normalizza_testo(cfor)

    if not k_cfor:
        continue

    if k_ceas and k_numero:
        mappa_fattura[(k_ceas, k_numero)].add(k_cfor)
    if k_ceas:
        mappa_ceas[k_ceas].add(k_cfor)


mappa_manuale = {
    normalizza_testo(chiave): normalizza_testo(valore)
    for chiave, valore in MAPPA_PIVA_CFOR.items()
    if normalizza_testo(chiave) and normalizza_testo(valore)
}


def attribuisci_cfor(ceas, numero_fattura, partita_iva, denominazione):
    """
    Restituisce (CFOR, origine). La mappa manuale ha priorita':
    e' una scelta esplicita di chi usa il notebook.
    """
    for chiave in (
        normalizza_testo(partita_iva),
        normalizza_testo(denominazione),
    ):
        if chiave in mappa_manuale:
            return mappa_manuale[chiave], "MAPPA_PIVA_CFOR"

    k_ceas = normalizza_testo(ceas)
    k_numero = normalizza_testo(numero_fattura)

    candidati = mappa_fattura.get((k_ceas, k_numero), set())
    if len(candidati) == 1:
        return next(iter(candidati)), "CARICHI_CEAS+FATTURA"
    if len(candidati) > 1:
        return sorted(candidati)[0], "CARICHI_AMBIGUO"

    candidati = mappa_ceas.get(k_ceas, set())
    if len(candidati) == 1:
        return next(iter(candidati)), "CARICHI_SOLO_CEAS"
    if len(candidati) > 1:
        return sorted(candidati)[0], "CARICHI_SOLO_CEAS_AMBIGUO"

    return "", "NON_TROVATO"


risultati_cfor = [
    attribuisci_cfor(
        riga["CEAS"],
        riga["NumeroFattura"],
        riga["PartitaIVA"],
        riga["Denominazione"],
    )
    for riga in df_documenti.to_dict("records")
]

df_documenti["CFOR"] = [cfor for cfor, _ in risultati_cfor]
df_documenti["OrigineCFOR"] = [
    origine for _, origine in risultati_cfor
]

df_codici = df_codici.merge(
    df_documenti[["ChiaveDocumento", "CFOR", "OrigineCFOR"]],
    on="ChiaveDocumento",
    how="left",
    validate="many_to_one",
)
df_codici["CFOR"] = df_codici["CFOR"].fillna("")


righe_fornitori = []
senza_cfor = df_documenti.loc[df_documenti["CFOR"] == ""]

for (piva, denominazione), gruppo in senza_cfor.groupby(
    ["PartitaIVA", "Denominazione"], dropna=False
):
    righe_fornitori.append({
        "PartitaIVA": piva or "",
        "Denominazione": denominazione or "",
        "Fatture": gruppo["ChiaveDocumento"].nunique(),
        "EsempioCEAS": ", ".join(
            sorted(set(gruppo["CEAS"].dropna()))[:2]
        ),
        "CFOR da compilare": "",
        "Nota": (
            "compilare il CFOR e riportarlo in MAPPA_PIVA_CFOR "
            "nella cella [1]"
        ),
    })

df_fornitori_da_mappare = pd.DataFrame(
    righe_fornitori,
    columns=[
        "PartitaIVA", "Denominazione", "Fatture",
        "EsempioCEAS", "CFOR da compilare", "Nota",
    ],
)

print(
    "Documenti con CFOR: "
    f"{int((df_documenti['CFOR'] != '').sum()):,} su "
    f"{len(df_documenti):,}"
)
print(
    "Origine del CFOR:",
    dict(Counter(df_documenti["OrigineCFOR"])),
)

if not df_fornitori_da_mappare.empty:
    print("\nFornitori senza CFOR (le loro righe non sono proponibili):")
    display(df_fornitori_da_mappare)

if "AMBIGUO" in " ".join(df_documenti["OrigineCFOR"]):
    print(
        "ATTENZIONE: fatture associate a più CFOR: la quadratura "
        "tiene il primo in ordine alfabetico, verificare a mano."
    )

## [4] Riscontro con anagrafiche e carichi

È il segnale più forte, ed è quello che rende il notebook utile anche quando il fornitore inventa un
nome di tipo sconosciuto: se i valori di quel tipo, una volta normalizzati, si ritrovano in `CEAN`,
il tipo è un EAN; se si ritrovano in `CART` è un codice cliente; se si ritrovano in `CARF`/`CALA`
è un codice fornitore.

Il confronto qui è **indicativo**: usa tre forme del valore (pulito, senza separatori/zeri, sole
cifre quando non ci sono lettere) e set di riferimento per `CFOR` quando disponibili. Non è il join
della quadratura, che resta governato dal dizionario. Per questo i conteggi sono usati come
*evidenza*, non come verità: un singolo valore che combacia per caso pesa poco.

In questa cella si calcolano anche le statistiche di forma, incluso il checksum GS1 (GTIN 8/12/13/14).

In [ ]:
# ==============================================================
# [4] RISCONTRO CON ANAGRAFICHE E CARICHI
# ==============================================================

def senza_separatori(valore):
    base = normalizza_testo(valore)
    if base is None:
        return None
    ridotto = "".join(c for c in base if c not in SEPARATORI)
    if ridotto and all(c in CIFRE for c in ridotto):
        ridotto = ridotto.lstrip("0") or "0"
    return ridotto or None


def solo_cifre(valore):
    base = normalizza_testo(valore)
    if base is None:
        return None
    # "ART002" -> "2" sarebbe un falso riscontro: si accetta
    # la forma solo-cifre solo se il valore non ha lettere.
    if any(c.isalpha() for c in base):
        return None
    cifre = "".join(c for c in base if c in CIFRE)
    return (cifre.lstrip("0") or "0") or None


def varianti(valore):
    """
    Forme usate SOLO per il riscontro indicativo. Il match vero
    della quadratura resta quello del dizionario.
    """
    base = normalizza_testo(valore)
    if base is None:
        return set()
    forme = {base}
    for funzione in (senza_separatori, solo_cifre):
        altra = funzione(valore)
        if altra:
            forme.add(altra)
    return forme


def ean_valido(valore):
    """GTIN 8/12/13/14 con checksum GS1 (modulo 10)."""
    base = normalizza_testo(valore)
    if (
        base is None
        or len(base) not in (8, 12, 13, 14)
        or any(c not in CIFRE for c in base)
    ):
        return False
    cifre = [int(c) for c in base]
    totale = sum(
        c * (3 if i % 2 == 0 else 1)
        for i, c in enumerate(reversed(cifre[:-1]))
    )
    return (10 - totale % 10) % 10 == cifre[-1]


def riferimenti(df, colonna_valore, colonna_cfor=None):
    """Set di forme normalizzate: globale e per CFOR."""
    globali = set()
    per_cfor = defaultdict(set)

    if df is not None and colonna_valore in df.columns:
        colonna_cfor_df = (
            df[colonna_cfor]
            if colonna_cfor and colonna_cfor in df.columns
            else None
        )

        for indice, valore in df[colonna_valore].items():
            forme = varianti(valore)
            if not forme:
                continue
            globali.update(forme)

            if colonna_cfor_df is not None:
                cfor = normalizza_testo(colonna_cfor_df[indice])
                if cfor:
                    per_cfor[cfor].update(forme)

    return {"globali": globali, "per_cfor": per_cfor}


df_ean = leggi_excel(EAN_FILE, EAN_SHEET)
df_carf = leggi_excel(CARF_FILE, CARF_SHEET)

rif_carf = riferimenti(df_carf, "CARF", "CFOR")
rif_cala = riferimenti(df_carichi, "CALA", "CFOR")

RIFERIMENTI = {
    "EAN": riferimenti(df_ean, "CEAN"),
    "CODICE_CLIENTE": riferimenti(df_carichi, "CART", "CFOR"),
    "CODICE_FORNITORE": {
        "globali": (
            rif_carf["globali"] | rif_cala["globali"]
        ),
        "per_cfor": {
            cfor: (
                rif_carf["per_cfor"].get(cfor, set())
                | rif_cala["per_cfor"].get(cfor, set())
            )
            for cfor in (
                set(rif_carf["per_cfor"])
                | set(rif_cala["per_cfor"])
            )
        },
    },
}


def insieme_riferimento(campo, cfor):
    """Set da usare per il riscontro; preferisce quello del CFOR."""
    riferimento = RIFERIMENTI[campo]
    if cfor and riferimento["per_cfor"].get(cfor):
        return riferimento["per_cfor"][cfor], "CFOR"
    return riferimento["globali"], "GLOBALE"


def forma_valore(valore):
    base = normalizza_testo(valore)
    if base is None:
        return {
            "SoloCifre": False, "Alfanumerico": False,
            "HaSeparatori": False, "ZeriIniziali": False,
            "Lunghezza": 0, "EANValido": False,
        }

    cifre = "".join(c for c in base if c in CIFRE)
    lunghezza = len(base)

    return {
        "SoloCifre": lunghezza > 0 and len(cifre) == lunghezza,
        "Alfanumerico": (
            any(c.isalpha() for c in base) and len(cifre) > 0
        ),
        "HaSeparatori": any(c in SEPARATORI for c in base),
        "ZeriIniziali": (
            base.startswith("0") and len(cifre) == lunghezza
        ),
        "Lunghezza": lunghezza,
        "EANValido": ean_valido(base),
    }


# Un riga per ogni valore distinto di ogni (CFOR, tipo).
df_codici["ValorePulito"] = [
    normalizza_testo(valore)
    for valore in df_codici["CodiceValoreXML"]
]

righe_valori = []
con_valore = df_codici.loc[df_codici["ValorePulito"].notna()]

for (cfor, tipo_chiave, pulito), gruppo in con_valore.groupby(
    ["CFOR", "TipoChiave", "ValorePulito"]
):
    forme = varianti(pulito)
    forma = forma_valore(pulito)

    riga = {
        "CFOR": cfor,
        "TipoChiave": tipo_chiave,
        "ValorePulito": pulito,
        "Occorrenze": len(gruppo),
        "EsempioOriginale": gruppo["CodiceValoreXML"].iloc[0],
    }

    for campo in CAMPI_CANONICI:
        insieme, _ = insieme_riferimento(campo, cfor)
        riga["Risc" + campo] = bool(forme & insieme) if insieme else False

    riga.update(forma)
    righe_valori.append(riga)

COLONNE_VALORI = (
    ["CFOR", "TipoChiave", "ValorePulito", "Occorrenze", "EsempioOriginale"]
    + ["Risc" + campo for campo in CAMPI_CANONICI]
    + [
        "SoloCifre", "Alfanumerico", "HaSeparatori",
        "ZeriIniziali", "Lunghezza", "EANValido",
    ]
)

df_valori = pd.DataFrame(righe_valori, columns=COLONNE_VALORI)

print(
    "Valori distinti analizzati: "
    f"{len(df_valori):,}"
)
print(
    "Forme di riferimento per campo:",
    {
        campo: len(RIFERIMENTI[campo]["globali"])
        for campo in CAMPI_CANONICI
    },
)

## [5] Segnali: nome del tipo e forma del valore

`ALIAS_GREZZI` è l'unico punto da aggiornare quando arriva un fornitore con un nome nuovo: è una
tabella di sinonimi, non una lista chiusa di regole. Se il nome non è in tabella si cercano parole
chiave (`EAN|GTIN|BARCODE`, `FORN|SUPPLIER|…`, `CLI|CUSTOMER|…`); se il nome contiene parole
riconducibili a due campi diversi diventa **ambiguo** e non assegna punti.

`TIPI_NOTI` raccoglie i tipi che *non* sono codici articolo (`MINSAN`, `SSCC`, `ISBN`, `LOTTO`,
`Descrizione`, …) e il tipo virtuale `RiferimentoAmministrazione`: vengono segnalati, non classificati
a priori. La forma del valore dà il secondo segnale: un GTIN con checksum valido pesa molto, una
stringa solo numerica di lunghezza compatibile pesa poco (molti codici interni sono numerici), un
codice alfanumerico non distingue cliente da fornitore.

`suggerisci_regola()` propone anche la `RegolaNormalizzazione`: separatori/zeri iniziali →
`RIMUOVI_SEPARATORI`, codice utile contenuto in una parte del valore → `ESTRAI_*`, valore con prefisso
testuale davanti a un riferimento numerico → `SOLO_NUMERI`. In mancanza di evidenze scrive
comunque il default in modo esplicito, così la stessa regola può essere proposta anche sul lato carico.

In [ ]:
# ==============================================================
# [5] SEGNALI: NOME DEL TIPO E FORMA DEL VALORE
# ==============================================================

def chiave_tipo(tipo):
    """Nome normalizzato: senza accenti, punteggiatura come spazio."""
    base = normalizza_testo(tipo) or ""
    base = "".join(
        c for c in unicodedata.normalize("NFKD", base)
        if not unicodedata.combining(c)
    )
    return " ".join(
        "".join(c if c.isalnum() else " " for c in base).split()
    )


def compatto_tipo(tipo):
    base = normalizza_testo(tipo) or ""
    return "".join(c for c in base if c.isalnum())


ALIAS_GREZZI = {
    "EAN": [
        "EAN", "EAN8", "EAN13", "EAN14", "EAN 8", "EAN 13", "EAN 14",
        "EAN-13", "EAN CODE", "CODICE EAN", "CODICE A BARRE",
        "BARCODE", "BAR CODE", "CODICE BARRE", "GTIN", "GTIN8",
        "GTIN12", "GTIN13", "GTIN14", "UPC", "UPC-A",
    ],
    "CODICE_FORNITORE": [
        "FORN", "FORNITORE", "COD F", "COD_F", "COD FORN",
        "CODICE FORNITORE", "CODICE ARTICOLO FORNITORE",
        "ARTICOLO FORNITORE", "ID FORNITORE", "SUPPLIER",
        "SUPPLIER CODE", "VENDOR", "VENDITORE", "PRODUTTORE",
        "MANUFACTURER", "MFR", "CARF", "CALA",
    ],
    "CODICE_CLIENTE": [
        "CLI", "CLIENTE", "COD CLI", "CODICE CLIENTE",
        "CODICE ARTICOLO CLIENTE", "ARTICOLO CLIENTE",
        "CUSTOMER", "CUSTOMER CODE", "ACQUIRENTE", "BUYER",
        "CART", "CODICE INTERNO", "ARTICOLO INTERNO", "INTERNO",
    ],
}

ALIAS_ESATTI = {
    chiave_tipo(alias): campo
    for campo, elenco in ALIAS_GREZZI.items()
    for alias in elenco
}

PAROLE_CHIAVE = (
    ("EAN", re.compile(r"EAN|GTIN|UPC|BAR ?CODE|BARRE")),
    ("CODICE_FORNITORE", re.compile(
        r"FORN|SUPPLIER|VENDOR|VENDITOR|PRODUTTOR|MFR|CARF|CALA"
    )),
    ("CODICE_CLIENTE", re.compile(
        r"CLI|CUSTOMER|ACQUIRENT|CART|INTERN|BUYER"
    )),
)

TIPI_NOTI = {
    "MINSAN": "codice ministeriale farmaco: non è EAN né codice cliente/fornitore",
    "AIC": "codice AIC farmaco: non è un codice articolo del gestionale",
    "SSCC": "codice collo/logistica",
    "ISBN": "codice editoriale",
    "ISSN": "codice editoriale",
    "LOTTO": "lotto di produzione",
    "SCADENZA": "data di scadenza",
    "DDT": "riferimento documento di trasporto",
    "ORDINE": "riferimento d'ordine",
    "DESCRIZIONE": (
        "tipo virtuale: la quadratura non crea più codici dalla descrizione"
    ),
    "RIFERIMENTOAMMINISTRAZIONE": (
        "tipo virtuale creato dalla quadratura: a volte è un codice articolo,"
        " va verificato sui valori"
    ),
    "SENZATIPO": (
        "CodiceTipo assente nell'XML: la chiave del dizionario non si può"
        " costruire, va corretto a monte"
    ),
}


def classifica_nome(tipo_chiave):
    """Restituisce (campo, peso, motivazione)."""
    if not tipo_chiave:
        return None, 0.0, TIPI_NOTI["SENZATIPO"]

    nome = chiave_tipo(tipo_chiave)

    if nome in ALIAS_ESATTI:
        campo = ALIAS_ESATTI[nome]
        return campo, 4.0, f"nome del tipo riconosciuto ({nome})"

    trovati = [
        campo for campo, regex in PAROLE_CHIAVE
        if regex.search(nome)
    ]

    if len(trovati) == 1:
        return (
            trovati[0], 2.5,
            f"parola chiave nel nome ({nome})",
        )

    if len(trovati) > 1:
        return (
            None, 0.0,
            "nome ambiguo: parole riconducibili a "
            + ", ".join(trovati),
        )

    nota = TIPI_NOTI.get(compatto_tipo(tipo_chiave))
    if nota:
        return None, 0.0, nota

    return None, 0.0, "nome del tipo non riconosciuto"


def classifica_forma(stats):
    """Restituisce (campo, peso, motivazione)."""
    if stats["quota_ean_valida"] >= 0.60:
        return (
            "EAN", 4.0,
            "valori con lunghezza e checksum GTIN validi",
        )

    if (
        stats["quota_solo_cifre"] >= 0.80
        and 6 <= stats["lunghezza_media"] <= 14
    ):
        return (
            "EAN", 1.5,
            "solo cifre in lunghezza GTIN (checksum non valido:"
            " indizio debole)",
        )

    if stats["quota_alfanumerico"] >= 0.70:
        return (
            None, 0.0,
            "codici alfanumerici: la forma non distingue"
            " cliente da fornitore",
        )

    return None, 0.0, "forma dei valori non discriminante"


def suggerisci_regola(valori_tipo, campo):
    """
    Regola proposta in base a come i valori XML si discostano dai
    valori di riferimento. Non modifica la quadratura: e' un testo
    che l'utente copia nel dizionario.
    """
    if campo not in CAMPI_CANONICI or valori_tipo.empty:
        return (
            "RIMUOVI_SEPARATORI",
            "default esplicito: coincide con il comportamento"
            " di una riga assente",
        )

    quota_sep = float(valori_tipo["HaSeparatori"].mean())
    quota_zeri = float(valori_tipo["ZeriIniziali"].mean())
    quota_cifre = float(valori_tipo["SoloCifre"].mean())
    cfor = valori_tipo["CFOR"].iloc[0]
    insieme, _ = insieme_riferimento(campo, cfor)
    totale = len(valori_tipo)

    if quota_sep >= 0.30 or quota_zeri >= 0.30:
        return (
            "RIMUOVI_SEPARATORI",
            "separatori o zeri iniziali presenti: rimozione"
            " da applicare su entrambi i lati",
        )

    primi = 0
    ultimi = 0
    for valore in valori_tipo["ValorePulito"]:
        pezzi = [
            p for p in re.split(r"[^A-Z0-9]+", valore) if p
        ]
        # Un valore di un solo pezzo non va "estratto": e' gia'
        # il codice intero. ESTRAI_* serve solo quando il codice
        # utile e' contenuto in un valore piu' lungo.
        if len(pezzi) < 2:
            continue
        if varianti(pezzi[0]) & insieme:
            primi += 1
        if varianti(pezzi[-1]) & insieme:
            ultimi += 1

    if quota_cifre < 0.50 and (primi or ultimi):
        if primi >= ultimi and primi >= max(1, totale * 0.40):
            return (
                "ESTRAI_PRIMA_STRINGA",
                "il primo pezzo del valore combacia con l'anagrafica",
            )
        if ultimi >= max(1, totale * 0.40):
            return (
                "ESTRAI_ULTIMA_STRINGA",
                "l'ultimo pezzo del valore combacia con l'anagrafica",
            )

    if (
        quota_cifre < 0.50
        and insieme
        and all(
            all(c in CIFRE for c in forma)
            for forma in insieme
        )
    ):
        return (
            "SOLO_NUMERI",
            "il valore contiene testo davanti a un riferimento"
            " numerico: verificare che le lettere siano solo un prefisso",
        )

    return (
        "RIMUOVI_SEPARATORI",
        "nessuna trasformazione particolare rilevata: default"
        " esplicito, da proporre identico sul lato carico",
    )


print(
    "Alias nome-tipo caricati:",
    len(ALIAS_ESATTI),
    "- tipi noti non articolo:",
    len(TIPI_NOTI),
)

## [6] Punteggio, classificazione e proposte

I tre segnali diventano un punteggio per ciascun `CampoCanonico`:

| Segnale | Peso |
|---|---|
| alias esatto sul nome | +4,0 |
| parola chiave sul nome | +2,5 |
| forma GTIN con checksum valido | +4,0 |
| forma solo numerica compatibile (checksum non valido) | +1,5 |
| riscontro forte in anagrafica (≥2 valori e ≥40% del tipo) | +5,0 (+1,5 se esclusivo) |
| riscontro debole (1 valore) | +2,0 (+0,5 per ogni valore in più) |
| concordanza nome ↔ riscontro | +1,5 |

Vince il punteggio più alto, ma servono almeno **2,5 punti** e un **margine di 1,5** sul secondo
classificato. Un nome che punta a un campo mentre l'anagrafica ne indica un altro è un conflitto:
il campo resta vuoto. Se il nome è sconosciuto e la classificazione poggia solo sul riscontro in
anagrafica, la confidenza massima è `MEDIA`. Le righe `BASSA` / `NESSUNA` non diventano `AGGIUNGI`.

Il risultato è `df_tipi`: una riga per `CFOR + CodiceTipoXML` con tutte le evidenze, i conteggi, i
punteggi e la `Motivazione` leggibile.

In [ ]:
# ==============================================================
# [6] PUNTEGGIO, CLASSIFICAZIONE E PROPOSTE
# ==============================================================

def statistiche_tipo(valori_tipo):
    if valori_tipo.empty:
        return {
            "n_valori": 0, "occorrenze": 0, "esempi": "",
            "quota_ean_valida": 0.0, "quota_solo_cifre": 0.0,
            "quota_alfanumerico": 0.0, "quota_separatori": 0.0,
            "lunghezza_media": 0.0,
        }

    esempi = list(
        dict.fromkeys(valori_tipo["EsempioOriginale"].tolist())
    )[:MAX_ESEMPI]

    return {
        "n_valori": len(valori_tipo),
        "occorrenze": int(valori_tipo["Occorrenze"].sum()),
        "esempi": " | ".join(str(e) for e in esempi),
        "quota_ean_valida": float(valori_tipo["EANValido"].mean()),
        "quota_solo_cifre": float(valori_tipo["SoloCifre"].mean()),
        "quota_alfanumerico": float(
            valori_tipo["Alfanumerico"].mean()
        ),
        "quota_separatori": float(
            valori_tipo["HaSeparatori"].mean()
        ),
        "lunghezza_media": float(valori_tipo["Lunghezza"].mean()),
    }


def riscontri_tipo(valori_tipo):
    """
    Per ogni campo: (peso, valori combacianti, quota, nota).
    Un solo valore che combacia per caso pesa poco.
    """
    esito = {}
    colonne = ["Risc" + campo for campo in CAMPI_CANONICI]
    totale = len(valori_tipo)

    for campo in CAMPI_CANONICI:
        colonna = "Risc" + campo
        altre = [c for c in colonne if c != colonna]
        n = int(valori_tipo[colonna].sum())
        quota = n / totale if totale else 0.0
        esclusivi = int(
            (
                valori_tipo[colonna]
                & ~valori_tipo[altre].any(axis=1)
            ).sum()
        )

        if n >= MIN_RISCONTRI and quota >= SOGLIA_RISCONTRO:
            peso = 5.0 + (1.5 if esclusivi == n else 0.0)
            nota = (
                f"{n}/{totale} valori trovati in {campo}"
                + (" (riscontro esclusivo)" if esclusivi == n else "")
            )
        elif n >= 1:
            peso = 2.0 + min(n - 1, 3) * 0.5
            nota = f"{n} valore/i trovati in {campo} (riscontro debole)"
        else:
            peso, nota = 0.0, "nessun riscontro"

        esito[campo] = (peso, n, quota, nota)

    return esito


def classifica_tipo(cfor, tipo_chiave, tipo_originale, valori_tipo):
    stats = statistiche_tipo(valori_tipo)
    riscontri = riscontri_tipo(valori_tipo)
    punteggi = {campo: 0.0 for campo in CAMPI_CANONICI}
    motivi = []

    nome_campo, peso_nome, nota_nome = classifica_nome(tipo_chiave)
    if nome_campo:
        punteggi[nome_campo] += peso_nome
    motivi.append("Nome: " + nota_nome + f" (+{peso_nome:g})")

    forma_campo, peso_forma, nota_forma = classifica_forma(stats)
    if forma_campo:
        punteggi[forma_campo] += peso_forma
    motivi.append("Forma: " + nota_forma + f" (+{peso_forma:g})")

    for campo, (peso, n, quota, nota) in riscontri.items():
        punteggi[campo] += peso
        if peso:
            motivi.append(
                f"Anagrafica {campo}: {nota} (+{peso:g})"
            )

    # Due segnali concordi valgono più della loro somma.
    if nome_campo and riscontri[nome_campo][0] > 0:
        punteggi[nome_campo] += 1.5
        motivi.append("Nome e riscontro concordi (+1.5)")

    miglior_riscontro = max(
        riscontri, key=lambda c: riscontri[c][0]
    )
    conflitto = bool(
        nome_campo
        and riscontri[miglior_riscontro][0] >= 2.0
        and miglior_riscontro != nome_campo
    )
    if (
        nome_campo and forma_campo
        and forma_campo != nome_campo
        and peso_forma >= 4.0
    ):
        conflitto = True

    if conflitto:
        motivi.append(
            "CONFLITTO: il nome indica "
            f"{nome_campo}, i dati indicano {miglior_riscontro}"
        )

    ordinati = sorted(
        punteggi.items(), key=lambda coppia: (-coppia[1], coppia[0])
    )
    migliore, primo = ordinati[0]
    secondo = ordinati[1][1]
    margine = primo - secondo

    if conflitto:
        campo, confidenza = "", "BASSA"
    elif primo < 2.5 or margine < MARGINE_MINIMO:
        campo, confidenza = "", "NESSUNA"
    elif primo >= SOGLIA_ALTA:
        campo = migliore
        confidenza = "ALTA" if nome_campo else "MEDIA"
    elif primo >= SOGLIA_MEDIA:
        campo, confidenza = migliore, "MEDIA"
    else:
        campo, confidenza = migliore, "BASSA"

    if campo:
        regola, nota_regola = suggerisci_regola(
            valori_tipo, campo
        )
    else:
        regola, nota_regola = "", (
            "nessuna regola proposta: prima va scelto il campo"
            " canonico"
        )

    return {
        "CampoCanonico": campo,
        "Confidenza": confidenza,
        "RegolaSuggerita": regola,
        "RegolaNormalizzazione": regola if campo else "",
        "NotaRegola": nota_regola,
        "Punteggio": round(primo, 1),
        "SecondoCampo": (
            ordinati[1][0] if secondo > 0 else ""
        ),
        "Margine": round(margine, 1),
        "Conflitto": conflitto,
        "Punteggi": "; ".join(
            f"{c}={p:g}" for c, p in ordinati
        ),
        "Motivazione": " | ".join(motivi),
        "Occorrenze": stats["occorrenze"],
        "ValoriDistinti": stats["n_valori"],
        "Esempi": stats["esempi"],
        "QuotaEANValida": round(stats["quota_ean_valida"], 2),
        "QuotaSoloCifre": round(stats["quota_solo_cifre"], 2),
        "QuotaAlfanumerico": round(
            stats["quota_alfanumerico"], 2
        ),
        "QuotaSeparatori": round(stats["quota_separatori"], 2),
        "RiscEAN": riscontri["EAN"][1],
        "RiscCODICE_CLIENTE": riscontri["CODICE_CLIENTE"][1],
        "RiscCODICE_FORNITORE": riscontri["CODICE_FORNITORE"][1],
    }


righe_tipi = []

for (cfor, tipo_chiave), gruppo in df_codici.groupby(
    ["CFOR", "TipoChiave"], dropna=False
):
    tipo_originale = Counter(
        gruppo["CodiceTipoXML"]
    ).most_common(1)[0][0]

    denominazioni = [
        d for d in gruppo["Denominazione"] if d
    ]
    denominazione = (
        Counter(denominazioni).most_common(1)[0][0]
        if denominazioni else ""
    )

    origini = set(gruppo["Origine"])
    origine = (
        "MISTO" if len(origini) > 1 else next(iter(origini))
    )

    valori_tipo = df_valori.loc[
        (df_valori["CFOR"] == cfor)
        & (df_valori["TipoChiave"] == tipo_chiave)
    ]

    esito = classifica_tipo(
        cfor, tipo_chiave, tipo_originale, valori_tipo
    )

    righe_tipi.append({
        "CFOR": cfor,
        "Denominazione": denominazione,
        "CodiceTipoXML": tipo_originale,
        "TipoChiave": tipo_chiave,
        "Origine": origine,
        "EsempioFileXML": gruppo["FileXML"].iloc[0],
        **esito,
    })

COLONNE_TIPI = [
    "CFOR", "Denominazione", "CodiceTipoXML", "TipoChiave",
    "Origine", "EsempioFileXML",
    "CampoCanonico", "Confidenza", "RegolaNormalizzazione",
    "RegolaSuggerita", "NotaRegola",
    "Occorrenze", "ValoriDistinti", "Esempi",
    "RiscEAN", "RiscCODICE_CLIENTE", "RiscCODICE_FORNITORE",
    "QuotaEANValida", "QuotaSoloCifre", "QuotaAlfanumerico",
    "QuotaSeparatori",
    "Punteggio", "SecondoCampo", "Margine", "Conflitto",
    "Punteggi", "Motivazione",
]

df_tipi = pd.DataFrame(righe_tipi, columns=COLONNE_TIPI)

print(f"Tipi osservati (CFOR + CodiceTipoXML): {len(df_tipi):,}")
print(
    "Classificati: "
    f"{int((df_tipi['CampoCanonico'] != '').sum()):,} - "
    "da verificare: "
    f"{int((df_tipi['CampoCanonico'] == '').sum()):,}"
)

## [7] Confronto con il dizionario esistente

Legge `Dizionario codici.xlsx` in **sola lettura** e costruisce le stesse chiavi della quadratura
(`CFOR + CodiceTipoXML` e `CFOR + CampoCanonico`, con `setdefault`: in caso di duplicato vale la
prima riga). Per ogni tipo osservato decide l'`Azione`:

| Azione | Significato |
|---|---|
| `AGGIUNGI` | tipo classificato con confidenza `ALTA`/`MEDIA` e assente dal dizionario |
| `GIA_PRESENTE` | riga già corretta, non serve copiarla |
| `VERIFICA_REGOLA` | stesso campo, ma la regola scritta differisce da quella suggerita |
| `VERIFICA_CONFLITTO` | il dizionario assegna un campo diverso da quello dedotto |
| `DA_VERIFICARE` | classificazione debole/assente: va nel foglio `DaVerificare` |
| `CFOR_MANCANTE` | serve prima mappare il fornitore (cella `[1]`) |

Vengono anche prodotte le **proposte simmetriche per `RegoleCodiciCarico`**: senza di esse il
notebook di quadratura segnala «regole diverse fra i due lati», perché una riga assente sul carico
ricade su `RIMUOVI_SEPARATORI` mentre `NESSUNA_REGOLA` mantiene i separatori. Infine si elencano le
regole configurate ma mai osservate negli XML e le anomalie (campo canonico o regola non riconosciuti,
chiavi duplicate).

La `RegolaNormalizzazione` suggerita vale **solo per le righe nuove**: su una riga già presente
l'assistente non propone di cambiarla, perché le regole dei due lati possono essere state scelte
apposta per quel fornitore. L'unica eccezione è una regola non riconosciuta, già segnalata fra le
anomalie.

In [ ]:
# ==============================================================
# [7] CONFRONTO CON IL DIZIONARIO ESISTENTE
# ==============================================================

try:
    fogli_dizionario = leggi_excel(DIZIONARIO_FILE, None)
except Exception as errore:
    print(
        "Dizionario non letto (si procede con proposte da zero):",
        errore,
    )
    fogli_dizionario = {}

if not isinstance(fogli_dizionario, dict):
    fogli_dizionario = {}

df_diz_xml = fogli_dizionario.get(NOME_FOGLIO_CODICI_XML)
df_diz_carico = fogli_dizionario.get(NOME_FOGLIO_CODICI_CARICO)


def mappa_regole_xml(df):
    """(CFOR, CODICETIPO) -> (CampoCanonico, regola)."""
    mappa = {}
    if not isinstance(df, pd.DataFrame):
        return mappa
    for _, riga in df.iterrows():
        cfor = normalizza_testo(riga.get("CFOR"))
        tipo = normalizza_testo(riga.get("CodiceTipoXML"))
        if not cfor or not tipo:
            continue
        mappa.setdefault((cfor, tipo), (
            normalizza_testo(riga.get("CampoCanonico")) or "",
            normalizza_testo(riga.get("RegolaNormalizzazione")) or "",
        ))
    return mappa


def mappa_regole_carico(df):
    """(CFOR, CAMPOCANONICO) -> regola."""
    mappa = {}
    if not isinstance(df, pd.DataFrame):
        return mappa
    for _, riga in df.iterrows():
        cfor = normalizza_testo(riga.get("CFOR"))
        campo = normalizza_testo(riga.get("CampoCanonico"))
        if not cfor or not campo:
            continue
        mappa.setdefault(
            (cfor, campo),
            normalizza_testo(riga.get("RegolaNormalizzazione")) or "",
        )
    return mappa


mappa_xml = mappa_regole_xml(df_diz_xml)
mappa_carico = mappa_regole_carico(df_diz_carico)


def anomalie_dizionario(df_xml, df_carico):
    """Refusi e chiavi duplicate, sullo schema usato dalla quadratura."""
    righe = []

    for foglio, df, chiavi, colonna_campo in (
        (
            NOME_FOGLIO_CODICI_XML, df_xml, set(),
            "CampoCanonico",
        ),
        (
            NOME_FOGLIO_CODICI_CARICO, df_carico, set(),
            "CampoCanonico",
        ),
    ):
        if not isinstance(df, pd.DataFrame):
            continue

        for _, riga in df.iterrows():
            cfor = normalizza_testo(riga.get("CFOR"))
            campo = normalizza_testo(
                riga.get(colonna_campo)
            ) or ""
            regola = normalizza_testo(
                riga.get("RegolaNormalizzazione")
            ) or ""
            valore_chiave = normalizza_testo(
                riga.get(
                    "CodiceTipoXML"
                    if foglio == NOME_FOGLIO_CODICI_XML
                    else "CampoCanonico"
                )
            ) or ""

            if not cfor or not valore_chiave:
                continue

            chiave = (foglio, cfor, valore_chiave)
            if chiave in chiavi:
                righe.append({
                    "Foglio": foglio, "CFOR": cfor,
                    "Chiave": valore_chiave, "Colonna": "-",
                    "Valore": "-",
                    "Problema": (
                        "chiave duplicata: vale la prima riga letta"
                    ),
                })
            chiavi.add(chiave)

            if campo and campo not in CAMPI_CANONICI:
                righe.append({
                    "Foglio": foglio, "CFOR": cfor,
                    "Chiave": valore_chiave,
                    "Colonna": "CampoCanonico", "Valore": campo,
                    "Problema": (
                        "CampoCanonico non ammesso: usare EAN,"
                        " CODICE_CLIENTE o CODICE_FORNITORE"
                    ),
                })

            if regola and regola not in REGOLE_NOTE:
                righe.append({
                    "Foglio": foglio, "CFOR": cfor,
                    "Chiave": valore_chiave,
                    "Colonna": "RegolaNormalizzazione",
                    "Valore": regola,
                    "Problema": (
                        "regola non riconosciuta: la quadratura"
                        " applica il default"
                    ),
                })

    return pd.DataFrame(
        righe,
        columns=[
            "Foglio", "CFOR", "Chiave", "Colonna",
            "Valore", "Problema",
        ],
    )


df_anomalie = anomalie_dizionario(df_diz_xml, df_diz_carico)


# --------------------------------------------------------------
# Azione per ogni tipo osservato
# --------------------------------------------------------------

campi_esistenti = []
regole_esistenti = []
stati = []
azioni = []

for _, riga in df_tipi.iterrows():
    presente = mappa_xml.get(
        (riga["CFOR"], riga["TipoChiave"])
    )
    campo = riga["CampoCanonico"]

    campi_esistenti.append(
        presente[0] if presente else ""
    )
    regole_esistenti.append(
        presente[1] if presente else ""
    )

    if not riga["CFOR"]:
        stato, azione = "CFOR_MANCANTE", "CFOR_MANCANTE"
    elif not campo:
        stato = "PRESENTE" if presente else "ASSENTE"
        azione = "DA_VERIFICARE"
    elif riga["Conflitto"]:
        stato = "PRESENTE" if presente else "ASSENTE"
        azione = "VERIFICA_CONFLITTO"
    elif riga["Confidenza"] == "BASSA":
        stato = "PRESENTE" if presente else "ASSENTE"
        azione = "DA_VERIFICARE"
    elif presente is None:
        stato, azione = "ASSENTE", "AGGIUNGI"
    elif presente[0] != campo:
        stato, azione = "PRESENTE_DIVERSO", "VERIFICA_CONFLITTO"
    elif presente[1] and presente[1] not in REGOLE_NOTE:
        # Una regola esistente NON viene riscritta: puo' essere
        # stata scelta apposta per questo fornitore. Si segnala
        # solo una regola che la quadratura non riconosce (ed e'
        # gia' elencata in AnomalieDizionario).
        stato = "PRESENTE_REGOLA_NON_RICONOSCIUTA"
        azione = "VERIFICA_REGOLA"
    else:
        stato, azione = "PRESENTE", "GIA_PRESENTE"

    stati.append(stato)
    azioni.append(azione)

df_tipi["CampoEsistente"] = campi_esistenti
df_tipi["RegolaEsistente"] = regole_esistenti
df_tipi["StatoDizionario"] = stati
df_tipi["Azione"] = azioni


COLONNE_META = [
    "Azione", "Confidenza", "StatoDizionario",
    "CampoEsistente", "RegolaEsistente", "Punteggio",
    "SecondoCampo", "Conflitto", "Punteggi",
    "Occorrenze", "ValoriDistinti", "Esempi",
    "Motivazione", "NotaRegola",
]

# Senza CFOR la riga non puo' essere copiata nel dizionario:
# resta fra i tipi da verificare anche se il campo e' stato dedotto.
proponibili = (
    (df_tipi["CampoCanonico"] != "")
    & (df_tipi["CFOR"] != "")
)
classificati = df_tipi.loc[proponibili].copy()
df_proposte_xml = classificati[
    COLONNE_PROPOSTA_XML + COLONNE_META
].reset_index(drop=True)

COLONNE_DA_VERIFICARE = [
    "CFOR", "Denominazione", "CodiceTipoXML", "Origine",
    "CampoCanonico", "RegolaNormalizzazione",
    "Azione", "Confidenza", "Conflitto", "Punteggi",
    "Occorrenze", "ValoriDistinti", "Esempi",
    "RiscEAN", "RiscCODICE_CLIENTE", "RiscCODICE_FORNITORE",
    "QuotaEANValida", "QuotaSoloCifre", "Motivazione",
]

df_da_verificare = df_tipi.loc[
    ~proponibili,
    COLONNE_DA_VERIFICARE,
].reset_index(drop=True)


# --------------------------------------------------------------
# Proposte simmetriche per RegoleCodiciCarico
# --------------------------------------------------------------

righe_carico = []

for (cfor, campo), gruppo in classificati.groupby(
    ["CFOR", "CampoCanonico"]
):
    if not cfor:
        continue

    regole = Counter(gruppo["RegolaNormalizzazione"])
    regola = regole.most_common(1)[0][0]
    nota = (
        ""
        if len(regole) == 1
        else "più regole suggerite per questo campo: verificare"
    )
    tipi = " | ".join(
        list(dict.fromkeys(gruppo["CodiceTipoXML"]))[:5]
    )
    esistente = mappa_carico.get((cfor, campo))

    if esistente is None:
        stato, azione = "ASSENTE", "AGGIUNGI"
    elif esistente and esistente not in REGOLE_NOTE:
        stato = "PRESENTE_REGOLA_NON_RICONOSCIUTA"
        azione = "VERIFICA_REGOLA"
    else:
        stato, azione = "PRESENTE", "GIA_PRESENTE"

    righe_carico.append({
        "CFOR": cfor,
        "Denominazione": gruppo["Denominazione"].iloc[0],
        "CampoCanonico": campo,
        "RegolaNormalizzazione": regola,
        "Azione": azione,
        "StatoDizionario": stato,
        "RegolaEsistente": esistente or "",
        "TipiCoinvolti": tipi,
        "Nota": nota,
    })

df_proposte_carico = pd.DataFrame(
    righe_carico,
    columns=COLONNE_PROPOSTA_CARICO + [
        "Azione", "StatoDizionario", "RegolaEsistente",
        "TipiCoinvolti", "Nota",
    ],
)


# --------------------------------------------------------------
# Regole configurate ma mai osservate
# --------------------------------------------------------------

osservate = set(zip(df_tipi["CFOR"], df_tipi["TipoChiave"]))
cfor_osservati = set(df_tipi["CFOR"])
righe_inutilizzate = []

for (cfor, tipo), (campo, regola) in mappa_xml.items():
    if (cfor, tipo) in osservate:
        continue
    nota = (
        "tipo configurato ma non osservato negli XML analizzati"
        if cfor in cfor_osservati
        else "fornitore assente dagli XML analizzati"
    )
    righe_inutilizzate.append({
        "Foglio": NOME_FOGLIO_CODICI_XML,
        "CFOR": cfor,
        "CodiceTipoXML": tipo,
        "CampoCanonico": campo,
        "RegolaNormalizzazione": regola,
        "Nota": nota,
    })

df_regole_inutilizzate = pd.DataFrame(
    righe_inutilizzate,
    columns=[
        "Foglio", "CFOR", "CodiceTipoXML", "CampoCanonico",
        "RegolaNormalizzazione", "Nota",
    ],
)

print(
    "Regole presenti nel dizionario: "
    f"XML={len(mappa_xml):,}, carico={len(mappa_carico):,}"
)
print(
    "Proposte XML:",
    len(df_proposte_xml),
    "- Azioni:",
    dict(Counter(df_proposte_xml["Azione"])),
)
print(
    "Da verificare:",
    len(df_da_verificare),
    "- proposte carico:",
    len(df_proposte_carico),
    "- regole inutilizzate:",
    len(df_regole_inutilizzate),
    "- anomalie:",
    len(df_anomalie),
)

## [8] Riepilogo, anteprime ed export delle proposte

Mostra le proposte con le **prime cinque colonne già nello schema di `RegoleCodiciXML`**, così si
possono copiare nel foglio dopo averle verificate. Le colonne dopo le prime cinque sono metadati
(`Azione`, `Confidenza`, `Motivazione`, …): non vanno copiate.

Se `ESPORTA_PROPOSTE = True` scrive `export/proposte_dizionario_codici.xlsx` (un foglio per ogni
tabella). **Non tocca mai `Dizionario codici.xlsx`**: la scelta finale resta umana.

Dopo aver copiato le righe `AGGIUNGI`, rieseguire `Quadratura_Fatture.ipynb` e controllare due
segnali: in `[12]` deve calare il numero di «Codici esclusi perché il tipo non è nel dizionario» e in
`[10]` non devono comparire `regole_non_allineate`.

In [ ]:
# ==============================================================
# [8] RIEPILOGO, ANTEPRIME ED EXPORT DELLE PROPOSTE
# ==============================================================

def mostra(tabella, colonne=None):
    if tabella is None or tabella.empty:
        print("(nessuna riga)")
        return
    display(
        tabella
        if colonne is None
        else tabella[
            [c for c in colonne if c in tabella.columns]
        ]
    )


righe_riepilogo = [
    {
        "Indicatore": "File XML con documenti FPR12",
        "Valore": df_documenti["FileXML"].nunique(),
    },
    {
        "Indicatore": "Documenti (body) analizzati",
        "Valore": len(df_documenti),
    },
    {
        "Indicatore": "Codici articolo letti",
        "Valore": len(df_codici),
    },
    {
        "Indicatore": "Tipi osservati (CFOR + CodiceTipoXML)",
        "Valore": len(df_tipi),
    },
    {
        "Indicatore": "Tipi classificati",
        "Valore": int((df_tipi["CampoCanonico"] != "").sum()),
    },
    {
        "Indicatore": "Righe AGGIUNGI proposte per RegoleCodiciXML",
        "Valore": int(
            (df_proposte_xml["Azione"] == "AGGIUNGI").sum()
        ),
    },
    {
        "Indicatore": "Righe AGGIUNGI proposte per RegoleCodiciCarico",
        "Valore": int(
            (df_proposte_carico["Azione"] == "AGGIUNGI").sum()
        )
        if not df_proposte_carico.empty else 0,
    },
    {
        "Indicatore": "Tipi da verificare",
        "Valore": len(df_da_verificare),
    },
    {
        "Indicatore": "Fornitori senza CFOR",
        "Valore": len(df_fornitori_da_mappare),
    },
    {
        "Indicatore": "Regole mai osservate negli XML",
        "Valore": len(df_regole_inutilizzate),
    },
    {
        "Indicatore": "Anomalie nel dizionario",
        "Valore": len(df_anomalie),
    },
    {
        "Indicatore": "XML non elaborati",
        "Valore": len(df_errori),
    },
]

df_riepilogo = pd.DataFrame(
    righe_riepilogo, columns=["Indicatore", "Valore"]
)

print("RIEPILOGO")
display(df_riepilogo)

print(
    "\nProposte per RegoleCodiciXML "
    "(solo le prime 5 colonne sono lo schema del dizionario):"
)
mostra(
    df_proposte_xml,
    COLONNE_PROPOSTA_XML
    + ["Azione", "Confidenza", "Motivazione"],
)

print("\nProposte per RegoleCodiciCarico:")
mostra(
    df_proposte_carico,
    COLONNE_PROPOSTA_CARICO + ["Azione", "TipiCoinvolti"],
)

print("\nTipi da verificare (nessuna proposta automatica):")
mostra(
    df_da_verificare,
    [
        "CFOR", "CodiceTipoXML", "CampoCanonico",
        "Azione", "Confidenza", "Punteggi", "Esempi",
        "Motivazione",
    ],
)


# --------------------------------------------------------------
# Export: MAI il dizionario originale
# --------------------------------------------------------------

if ESPORTA_PROPOSTE:

    if MODALITA_LOCALE:
        os.makedirs(CARTELLA_LOCALE, exist_ok=True)
        file_export = f"{CARTELLA_LOCALE}/{OUTPUT_FILE}"
    else:
        cartella_export = f"{cartella}/export"
        if not notebookutils.fs.exists(cartella_export):
            notebookutils.fs.mkdirs(cartella_export)
        file_export = f"{cartella_export}/{OUTPUT_FILE}"


    def motore_excel():
        try:
            import xlsxwriter  # noqa: F401
            return "xlsxwriter"
        except ImportError:
            return "openpyxl"


    TABELLE_DA_ESPORTARE = {
        "ProposteXML": df_proposte_xml,
        "ProposteCarico": df_proposte_carico,
        "DaVerificare": df_da_verificare,
        "DettaglioTipi": df_tipi,
        "FornitoriDaMappare": df_fornitori_da_mappare,
        "RegoleInutilizzate": df_regole_inutilizzate,
        "AnomalieDizionario": df_anomalie,
        "Riepilogo": df_riepilogo,
    }

    with pd.ExcelWriter(
        file_export, engine=motore_excel()
    ) as writer:
        for nome_foglio, tabella in TABELLE_DA_ESPORTARE.items():
            if tabella is None or tabella.empty:
                print(
                    f"Nessun dato da esportare per: {nome_foglio}"
                )
                continue

            tabella.to_excel(
                writer,
                sheet_name=nome_foglio[:31],
                index=False,
            )
            print(
                f"Esportato {nome_foglio}: "
                f"{len(tabella):,} righe, "
                f"{len(tabella.columns)} colonne"
            )

    print("\nFile salvato:", file_export)

else:
    print(
        "Export disattivato "
        "(ESPORTA_PROPOSTE = False): le tabelle restano in memoria."
    )

print(
    "\nIl dizionario originale NON è stato modificato: "
    "verificare le proposte prima di copiarle."
)